# EgoBrain — Neural-Temporal Adaptive Routing Reproduction

**Purpose:** reproduce the completed EEG-privileged adaptive temporal routing experiment and its negative controls using the locked visual benchmark.

### Important scientific status
This is the exact completed routing experiment recorded for the project. Its final gate was **INCONCLUSIVE**: the 50% temporal budget reduced estimated temporal FLOPs by 50.16%, but the final visual-only deployment did not preserve clean recognition or mean corruption Macro-F1 relative to the frozen visual baseline, and the neural control effect was not established.

The notebook is therefore retained as a reproducibility record, not as a claimed accuracy-improvement method.

## Scientific contract

The notebook does **not** modify or retrain Phase 2. It loads the frozen Phase-2 checkpoint and exact split.

The EEG branch is training-only. The final model is tested with:
- no EEG tensor,
- no EEG preprocessing,
- no EEG encoder,
- no EEG latency.

The Phase-3 claim is only considered established when clean recognition is preserved, temporal robustness improves, temporal-model computation decreases, and the effect weakens under neural negative controls.

In [ ]:

# ============================================================
# CELL 1 — CONFIGURATION / REPRODUCIBILITY LOCK
# ============================================================
from pathlib import Path
import os, json, math, time, random, shutil, hashlib, warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 20260928
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
try: torch.set_float32_matmul_precision("high")
except Exception: pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

SUBJECTS = [f"P{i:04d}" for i in range(1,41)]
VISUAL_DIM, EEG_DIM, SEQ_LEN, NUM_CLASSES = 512, 128, 8, 27
TRAIN_STRIDE, EVAL_STRIDE = 4, 1

# Exact Phase-2 visual Transformer hyperparameters
D_MODEL, HEADS, LAYERS, FF_DIM, DROPOUT = 256, 8, 4, 512, 0.10

# EEG teacher
EEG_D_MODEL, EEG_HEADS, EEG_LAYERS, EEG_FF_DIM, EEG_DROPOUT = 128, 4, 3, 256, 0.10

TEACHER_BATCH = 192
VISION_BATCH = 128
NUM_WORKERS = 0
WEIGHT_DECAY, GRAD_CLIP = 1e-4, 1.0

TEACHER_EPOCHS, TEACHER_PATIENCE = 10, 3
RELATION_EPOCHS, RELATION_PATIENCE = 8, 3
IMPORTANCE_EPOCHS, IMPORTANCE_PATIENCE = 6, 3
ROUTER_EPOCHS, ROUTER_PATIENCE = 5, 2
BUDGET_EPOCHS, BUDGET_PATIENCE = 5, 2
ROBUST_EPOCHS, ROBUST_PATIENCE = 4, 2
CONTROL_EPOCHS, CONTROL_PATIENCE = 4, 2

EEG_LR, VISION_LR, ROUTER_LR = 3e-4, 3e-4, 2e-4
LAMBDA_RELATION, LAMBDA_IMPORTANCE, LAMBDA_BUDGET, LAMBDA_ROBUST = 0.50, 0.50, 0.20, 0.25
ROUTER_SOFT_TEMPERATURE = 1.0

PRIMARY_BUDGET = 0.50
BUDGETS = [1.00, 0.75, 0.50, 0.25, 0.125]
CORRUPTION_SEVERITIES = [0.05, 0.15, 0.30, 0.50, 0.70]
CORRUPTION_NAMES = ["random_frame_drop","burst_deletion","temporal_jitter","irregular_sampling","playback_speed_change","contiguous_blackout"]

# Predeclared gate margins; not tuned on the test set.
CLEAN_TOL_ABS = 0.002
ROBUSTNESS_GAIN_MIN = 0.001
CONTROL_WEAKENING_MIN = 0.002
BOOTSTRAP_ITERS = 2000

RUN_STAGE_31_EEG_TEACHER = True
RUN_STAGE_33_RELATION = True
RUN_STAGE_34_IMPORTANCE = True
RUN_STAGE_35_ROUTER = True
RUN_STAGE_36_BUDGET = True
RUN_STAGE_36_ROBUSTNESS = True
RUN_NEGATIVE_CONTROLS = True
RUN_BOOTSTRAP = True
RESUME_PHASE3 = True

SEMANTIC_TASK_CLASSES = [
    "draw_computer_pictures","draw_pictures","draw_tracing",
    "drink_bitter_gourd_juice","drink_coca_cola","drink_water",
    "eat_snack","eat_spicy_strips","eat_sugers",
    "operate_computer_excel","operate_computer_powerpoint","operate_computer_word",
    "play_cards","play_computer_game_keyboard","play_computer_game_mouse",
    "play_cube","play_phone_game_complex","play_phone_game_simple",
    "play_puzzle","play_toy","read_comic_book","read_research_arcticle",
    "read_textbook","watch_monitor","watch_smartphone","write_notes","write_tracing",
]
CLASS_TO_ID = {c:i for i,c in enumerate(SEMANTIC_TASK_CLASSES)}
ID_TO_CLASS = {i:c for c,i in CLASS_TO_ID.items()}

PHASE1_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-egobrain-actual-phase-1-output"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase-1-output/egobrain_phase1"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase-1-output"),
]
VISUAL_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-egobrain-visual-features/uniform"),
]
EEG_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-eeg-features/eeg_features"),
    Path("/kaggle/input/datasets/keerthikumarkj/eeg_features"),
]
PHASE2_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-phase-2-actual-outputs"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase2-actual-outputs"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain_phase2_artifacts"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase2"),
]

OUT_ROOT = Path("/kaggle/working/egobrain_phase3_neuroroute")
MODEL_ROOT, CACHE_ROOT = OUT_ROOT/"models", OUT_ROOT/"teacher_cache"
REPORT_ROOT, FIG_ROOT = OUT_ROOT/"reports", OUT_ROOT/"figures"
CONTROL_ROOT, DEPLOY_ROOT = OUT_ROOT/"controls", OUT_ROOT/"deployment"
for p in [OUT_ROOT,MODEL_ROOT,CACHE_ROOT,REPORT_ROOT,FIG_ROOT,CONTROL_ROOT,DEPLOY_ROOT]: p.mkdir(parents=True, exist_ok=True)
print("Primary budget:", PRIMARY_BUDGET, "=>", max(1,int(round(SEQ_LEN*PRIMARY_BUDGET))), "tokens")


In [ ]:

# ============================================================
# CELL 2 — RESOLVE FROZEN PHASE-1 / PHASE-2 INPUTS
# ============================================================
def first_existing(paths):
    for p in paths:
        if p.exists(): return p
    return None

def find_file(root, names):
    if root is None: return None
    for name in names:
        p = root/name
        if p.exists(): return p
    for name in names:
        hits = sorted(root.rglob(name))
        if hits: return hits[0]
    return None

PHASE1_ROOT = first_existing(PHASE1_ROOT_CANDIDATES)
VISUAL_ROOT = first_existing(VISUAL_ROOT_CANDIDATES)
EEG_ROOT = first_existing(EEG_ROOT_CANDIDATES)
PHASE2_ROOT = first_existing(PHASE2_ROOT_CANDIDATES)

if PHASE1_ROOT is None: raise FileNotFoundError("Phase-1 root not found.")
if VISUAL_ROOT is None: raise FileNotFoundError("512-D visual feature root not found.")
if EEG_ROOT is None: raise FileNotFoundError("128-D EEG feature root not found.")
if PHASE2_ROOT is None: raise FileNotFoundError("Phase-2 artifact root not found.")

LABELED_INDEX = find_file(PHASE1_ROOT, ["all_subjects_synchronized_labeled_index.csv"])
PHASE2_SPLIT = find_file(PHASE2_ROOT, ["sequence_index_with_split.csv"])
PHASE2_MEAN = find_file(PHASE2_ROOT, ["train_feature_mean.npy"])
PHASE2_STD = find_file(PHASE2_ROOT, ["train_feature_std.npy"])
PHASE2_WEIGHTS = find_file(PHASE2_ROOT, ["class_weights.pt"])
PHASE2_CLEAN = find_file(PHASE2_ROOT, ["clean_model_summary.csv"])
PHASE2_CORRUPTIONS = find_file(PHASE2_ROOT, ["corruption_benchmark.csv"])

checkpoint_candidates = [
    PHASE2_ROOT/"models"/"temporal_transformer"/"best_checkpoint.pt",
    PHASE2_ROOT/"models"/"temporal_transformer"/"best_weights_only.pt",
]
PHASE2_CHECKPOINT = first_existing(checkpoint_candidates)
if PHASE2_CHECKPOINT is None:
    for pat in ["best_checkpoint.pt","best_weights_only.pt"]:
        hits = [p for p in sorted(PHASE2_ROOT.rglob(pat)) if "temporal_transformer" in str(p)]
        if hits: PHASE2_CHECKPOINT = hits[0]; break

if LABELED_INDEX is None: raise FileNotFoundError("Frozen Phase-1 labeled index missing.")
if PHASE2_SPLIT is None: raise FileNotFoundError("Frozen Phase-2 sequence_index_with_split.csv missing.")
if PHASE2_CHECKPOINT is None: raise FileNotFoundError("Canonical temporal_transformer checkpoint missing.")

print("PHASE1_ROOT:", PHASE1_ROOT)
print("VISUAL_ROOT:", VISUAL_ROOT)
print("EEG_ROOT:", EEG_ROOT)
print("PHASE2_ROOT:", PHASE2_ROOT)
print("LABELED_INDEX:", LABELED_INDEX)
print("PHASE2_SPLIT:", PHASE2_SPLIT)
print("PHASE2_CHECKPOINT:", PHASE2_CHECKPOINT)


## 1. Frozen split and multimodal join

Neural-route reuses the exact Phase-2 sequence index. The EEG feature start index is recovered from the Phase-1 synchronized index, so the same 8 seconds are read from both modalities.

In [ ]:

# ============================================================
# CELL 3 — LOAD FROZEN SPLIT + EEG INDEX MAPPING
# ============================================================
phase1_index = pd.read_csv(LABELED_INDEX)
seq_df = pd.read_csv(PHASE2_SPLIT)

req1 = {"subject_id","sync_index","visual_feature_idx","eeg_feature_idx","start_sec","end_sec","action_label"}
req2 = {"subject_id","video_source_order","action_label","label_id","sync_start","sync_end","visual_start","visual_end","start_sec","end_sec","split"}
if req1-set(phase1_index.columns): raise RuntimeError(f"Phase-1 columns missing: {sorted(req1-set(phase1_index.columns))}")
if req2-set(seq_df.columns): raise RuntimeError(f"Phase-2 sequence columns missing: {sorted(req2-set(seq_df.columns))}")

seq_df = seq_df[seq_df.split.isin(["train","val","test"])].reset_index(drop=True)
train_seq = seq_df[seq_df.split=="train"].reset_index(drop=True)
val_seq = seq_df[seq_df.split=="val"].reset_index(drop=True)
test_seq = seq_df[seq_df.split=="test"].reset_index(drop=True)

train_subjects = sorted(train_seq.subject_id.unique().tolist())
val_subjects = sorted(val_seq.subject_id.unique().tolist())
test_subjects = sorted(test_seq.subject_id.unique().tolist())
assert len(train_subjects)==24 and len(val_subjects)==8 and len(test_subjects)==8
assert not(set(train_subjects)&set(val_subjects)|set(train_subjects)&set(test_subjects)|set(val_subjects)&set(test_subjects))

expected = {"train":31147,"val":38404,"test":46178}
actual = {"train":len(train_seq),"val":len(val_seq),"test":len(test_seq)}
print("Frozen Phase-2 counts:", actual)
if actual != expected:
    raise RuntimeError(f"Frozen Phase-2 split count mismatch: expected {expected}, got {actual}")

eeg_lookup = phase1_index.set_index(["subject_id","sync_index"])["eeg_feature_idx"]
def attach_eeg_start(table):
    starts=[]
    for r in table.itertuples(index=False):
        key=(r.subject_id,int(r.sync_start))
        if key not in eeg_lookup.index: raise RuntimeError(f"Missing EEG mapping: {key}")
        starts.append(int(eeg_lookup.loc[key]))
    out=table.copy(); out["eeg_start"]=starts
    return out

train_seq=attach_eeg_start(train_seq)
val_seq=attach_eeg_start(val_seq)
test_seq=attach_eeg_start(test_seq)

print("TRAIN:", train_subjects)
print("VAL:", val_subjects)
print("TEST:", test_subjects)


## 2. Load the frozen visual and EEG feature tensors

In [ ]:

# ============================================================
# CELL 4 — LOAD VISUAL + EEG FEATURES
# ============================================================
def load_tensor_file(path):
    obj=torch.load(path,map_location="cpu",weights_only=False)
    if isinstance(obj,dict) and "features" in obj: return obj["features"],obj
    return obj,{}

def find_subject_file(root, subject, preferred):
    for name in preferred:
        p=root/name
        if p.exists(): return p
    hits=sorted(root.glob(f"*{subject}*.pt"))
    if not hits: raise FileNotFoundError(f"No feature file for {subject} in {root}")
    return hits[0]

visual_cache={}; eeg_cache={}; rows=[]
for s in SUBJECTS:
    vp=find_subject_file(VISUAL_ROOT,s,[f"features_{s}.pt"])
    vp_obj,vmeta=load_tensor_file(vp)
    if vp_obj.ndim!=2 or vp_obj.shape[1]!=VISUAL_DIM: raise RuntimeError(f"{s}: visual shape {tuple(vp_obj.shape)}")
    if not torch.isfinite(vp_obj.float()).all(): raise RuntimeError(f"{s}: visual NaN/Inf")
    visual_cache[s]=vp_obj.cpu()

    ep=find_subject_file(EEG_ROOT,s,[f"eeg_{s}.pt",f"features_{s}.pt"])
    ep_obj,emeta=load_tensor_file(ep)
    if ep_obj.ndim!=2 or ep_obj.shape[1]!=EEG_DIM: raise RuntimeError(f"{s}: EEG shape {tuple(ep_obj.shape)}")
    if not torch.isfinite(ep_obj.float()).all(): raise RuntimeError(f"{s}: EEG NaN/Inf")
    eeg_cache[s]=ep_obj.float().cpu()

    rows.append({"subject_id":s,"visual_rows":len(vp_obj),"eeg_rows":len(ep_obj),"visual_path":str(vp),"eeg_path":str(ep)})

inventory=pd.DataFrame(rows)
inventory.to_csv(REPORT_ROOT/"feature_inventory.csv",index=False)
display(inventory)


In [ ]:
# ============================================================
# CELL 5 — MULTIMODAL WINDOW INTEGRITY (FAST VERIFICATION)
# ============================================================
def validate_table_fast(table, name, sample_size=200):
    # Verify overall tensor boundaries
    for r in table.itertuples(index=False):
        s = r.subject_id; v0 = int(r.visual_start); e0 = int(r.eeg_start)
        if visual_cache[s][v0:v0+SEQ_LEN].shape != (SEQ_LEN, VISUAL_DIM):
            raise RuntimeError(f"{name} visual slice bad: {s}:{v0}")
        if eeg_cache[s][e0:e0+SEQ_LEN].shape != (SEQ_LEN, EEG_DIM):
            raise RuntimeError(f"{name} EEG slice bad: {s}:{e0}")

    # Spot-check temporal continuity on a random subset
    sample = table.sample(n=min(sample_size, len(table)), random_state=SEED)
    for r in sample.itertuples(index=False):
        s = r.subject_id
        q = phase1_index[(phase1_index.subject_id == s) & phase1_index.sync_index.between(int(r.sync_start), int(r.sync_end))].sort_values("sync_index")
        if len(q) != SEQ_LEN or not np.all(np.diff(q.sync_index.to_numpy()) == 1):
            raise RuntimeError(f"{name}: temporal continuity gap at {s}:{r.sync_start}")

    print("PASS", name, len(table))

for nm, tb in [("train", train_seq), ("val", val_seq), ("test", test_seq)]:
    validate_table_fast(tb, nm)

## 3. Reuse Phase-2 visual normalization; compute EEG normalization from train subjects only

In [ ]:

# ============================================================
# CELL 6 — NORMALIZATION + CLASS WEIGHTS
# ============================================================
if PHASE2_MEAN is None or PHASE2_STD is None: raise FileNotFoundError("Phase-2 train visual statistics missing")
visual_mean=torch.from_numpy(np.load(PHASE2_MEAN)).float()
visual_std=torch.from_numpy(np.load(PHASE2_STD)).float().clamp_min(1e-6)

if PHASE2_WEIGHTS is not None:
    class_weights=torch.load(PHASE2_WEIGHTS,map_location="cpu",weights_only=False).float()
else:
    c=train_seq.label_id.value_counts().reindex(range(NUM_CLASSES),fill_value=0).to_numpy(dtype=np.float32)
    ct=torch.tensor(c); class_weights=(ct.max()/ct).pow(0.5); class_weights/=class_weights.mean()

sumv=torch.zeros(EEG_DIM,dtype=torch.float64); sumsq=torch.zeros(EEG_DIM,dtype=torch.float64); n=0
for s in train_subjects:
    x=eeg_cache[s].double(); sumv+=x.sum(0); sumsq+=(x*x).sum(0); n+=len(x)
eeg_mean=(sumv/n).float()
eeg_var=(sumsq/n-eeg_mean.double()**2).clamp_min(1e-8)
eeg_std=torch.sqrt(eeg_var).float().clamp_min(1e-6)
np.save(OUT_ROOT/"train_eeg_mean.npy",eeg_mean.numpy())
np.save(OUT_ROOT/"train_eeg_std.npy",eeg_std.numpy())
print("Phase-2 visual stats reused; EEG stats from",n,"train-subject EEG rows.")


# Stage 3.1 — EEG temporal teacher

The teacher learns `zE(t)` and an action-functional temporal importance `aE(t)`. The importance distribution is the actual pooling mechanism used by the EEG classifier, so it is not a post-hoc visualization score.

In [ ]:
# ============================================================
# CELL 7 — MODEL UTILITIES + EEG TEACHER
# ============================================================
from contextlib import nullcontext

def autocast_context():
    if DEVICE.type!="cuda": return nullcontext()
    try: return torch.autocast(device_type="cuda",dtype=torch.float16)
    except Exception: return torch.cuda.amp.autocast()

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=SEQ_LEN):
        super().__init__()
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        # FIX: Add the batch dimension to the positional encoding buffer
        self.register_buffer("pe",pe.unsqueeze(0),persistent=False)
        
    def forward(self,x,positions=None):
        if positions is None: positions=torch.arange(x.shape[1],device=x.device)
        # FIX: Properly index the temporal dimension while broadcasting the batch
        if positions.ndim==1: return x+self.pe[:,positions,:]
        # Handles 2D adaptive routing indices [B, T]
        return x+self.pe[0][positions]

class EEGTemporalTeacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.input=nn.Linear(EEG_DIM,EEG_D_MODEL)
        self.pos=SinusoidalPositionalEncoding(EEG_D_MODEL)
        layer=nn.TransformerEncoderLayer(d_model=EEG_D_MODEL,nhead=EEG_HEADS,dim_feedforward=EEG_FF_DIM,dropout=EEG_DROPOUT,activation="gelu",batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=EEG_LAYERS)
        self.norm=nn.LayerNorm(EEG_D_MODEL)
        self.importance_head=nn.Linear(EEG_D_MODEL,1)
        self.classifier=nn.Linear(EEG_D_MODEL,NUM_CLASSES)
    def forward(self,x):
        z=self.norm(self.encoder(self.pos(self.input(x))))
        a_logits=self.importance_head(z).squeeze(-1)
        a=torch.softmax(a_logits,dim=1)
        pooled=(z*a.unsqueeze(-1)).sum(1)
        return {"tokens":z,"importance":a,"importance_logits":a_logits,"pooled":pooled,"logits":self.classifier(pooled)}

def compute_metrics(y,p):
    return {"accuracy":float(accuracy_score(y,p)),"balanced_accuracy":float(balanced_accuracy_score(y,p)),
            "macro_f1":float(f1_score(y,p,average="macro",zero_division=0)),
            "weighted_f1":float(f1_score(y,p,average="weighted",zero_division=0))}

def save_ckpt(path,model,optimizer=None,epoch=0,best_metric=-np.inf,history=None,extra=None):
    payload={"model_state":model.state_dict(),"epoch":epoch,"best_metric":float(best_metric),"history":history or []}
    if optimizer is not None: payload["optimizer_state"]=optimizer.state_dict()
    if extra is not None: payload["extra"]=extra
    torch.save(payload,path)

def load_state(path,model,strict=True):
    payload=torch.load(path,map_location="cpu",weights_only=False)
    state=payload.get("best_model_state") if isinstance(payload,dict) and payload.get("best_model_state") is not None else payload.get("model_state") if isinstance(payload,dict) and "model_state" in payload else payload
    if not isinstance(state,dict): raise RuntimeError(f"Unsupported checkpoint {path}")
    state={k[7:] if k.startswith("module.") else k:v for k,v in state.items()}
    missing,unexpected=model.load_state_dict(state,strict=strict)
    if missing or unexpected: raise RuntimeError(f"Checkpoint mismatch missing={missing}, unexpected={unexpected}")
    return payload

teacher=EEGTemporalTeacher().to(DEVICE)
p=teacher(torch.randn(2,SEQ_LEN,EEG_DIM,device=DEVICE))
assert p["logits"].shape==(2,NUM_CLASSES)
assert p["importance"].shape==(2,SEQ_LEN)
print("PASS EEG teacher smoke test")


In [ ]:

# ============================================================
# CELL 8 — EEG TEACHER DATASET + TRAINING
# ============================================================
class EEGSequenceDataset(Dataset):
    def __init__(self,table):
        self.df=table.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        x=eeg_cache[r.subject_id][int(r.eeg_start):int(r.eeg_start)+SEQ_LEN].float()
        x=(x-eeg_mean)/eeg_std
        return x,int(r.label_id),i

train_eeg_ds=EEGSequenceDataset(train_seq); val_eeg_ds=EEGSequenceDataset(val_seq); test_eeg_ds=EEGSequenceDataset(test_seq)
dl_kwargs=dict(batch_size=TEACHER_BATCH,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=="cuda"))
train_eeg_loader=DataLoader(train_eeg_ds,shuffle=True,**dl_kwargs)
val_eeg_loader=DataLoader(val_eeg_ds,shuffle=False,**dl_kwargs)
test_eeg_loader=DataLoader(test_eeg_ds,shuffle=False,**dl_kwargs)

def eval_teacher(model,loader,return_outputs=False):
    model.eval(); ys=[]; ps=[]; imp_by_idx=[None]*len(loader.dataset); tok_by_idx=[None]*len(loader.dataset)
    with torch.no_grad():
        for xb,yb,idx in loader:
            xb=xb.to(DEVICE)
            with autocast_context(): o=model(xb)
            ys.extend(yb.numpy().tolist()); ps.extend(o["logits"].argmax(1).cpu().numpy().tolist())
            if return_outputs:
                for j,k in enumerate(idx.numpy()):
                    imp_by_idx[int(k)]=o["importance"][j].float().cpu()
                    tok_by_idx[int(k)]=o["tokens"][j].float().cpu()
    m=compute_metrics(np.asarray(ys),np.asarray(ps))
    if return_outputs: return m,torch.stack(imp_by_idx),torch.stack(tok_by_idx)
    return m

teacher_ckpt=MODEL_ROOT/"stage31_eeg_teacher.pt"
teacher_hist=REPORT_ROOT/"stage31_eeg_teacher_history.csv"
criterion=nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer=torch.optim.AdamW(teacher.parameters(),lr=EEG_LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=TEACHER_EPOCHS)

start=0; best=-np.inf; best_state=None; history=[]
if RESUME_PHASE3 and teacher_ckpt.exists():
    payload=torch.load(teacher_ckpt,map_location="cpu",weights_only=False)
    teacher.load_state_dict(payload["model_state"]); start=int(payload.get("epoch",0))+1
    best=float(payload.get("best_metric",-np.inf)); history=payload.get("history",[])
    if "optimizer_state" in payload: optimizer.load_state_dict(payload["optimizer_state"])
    print("RESUME teacher:",start)

if start<TEACHER_EPOCHS:
    patience=0
    scaler=torch.amp.GradScaler("cuda") if DEVICE.type=="cuda" else None
    for epoch in range(start,TEACHER_EPOCHS):
        teacher.train(); total=0.; seen=0
        for xb,yb,_ in train_eeg_loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); optimizer.zero_grad(set_to_none=True)
            with autocast_context(): loss=criterion(teacher(xb)["logits"],yb)
            if scaler:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(teacher.parameters(),GRAD_CLIP); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); nn.utils.clip_grad_norm_(teacher.parameters(),GRAD_CLIP); optimizer.step()
            total+=float(loss.detach().cpu())*len(yb); seen+=len(yb)
        scheduler.step()
        vm=eval_teacher(teacher,val_eeg_loader)
        row={"epoch":epoch,"train_loss":total/max(seen,1),**{f"val_{k}":v for k,v in vm.items()},"lr":optimizer.param_groups[0]["lr"]}
        history.append(row); pd.DataFrame(history).to_csv(teacher_hist,index=False)
        print(f"teacher ep {epoch+1:02d} | loss {row['train_loss']:.4f} | val F1 {row['val_macro_f1']:.4f}")
        if row["val_macro_f1"]>best+1e-6:
            best=row["val_macro_f1"]; best_state={k:v.detach().cpu().clone() for k,v in teacher.state_dict().items()}; patience=0
        else: patience+=1
        save_ckpt(teacher_ckpt,teacher,optimizer,epoch,best,history)
        if patience>=TEACHER_PATIENCE: break
    if best_state is not None: teacher.load_state_dict(best_state)

teacher.eval()
teacher_val_metrics=eval_teacher(teacher,val_eeg_loader)
teacher_test_metrics,test_teacher_imp,test_teacher_tokens=eval_teacher(teacher,test_eeg_loader,True)
teacher_train_metrics=eval_teacher(teacher,train_eeg_loader)
teacher_summary=pd.DataFrame([
    {"split":"train",**teacher_train_metrics},{"split":"val",**teacher_val_metrics},{"split":"test",**teacher_test_metrics}
])
teacher_summary.to_csv(REPORT_ROOT/"stage31_eeg_teacher_metrics.csv",index=False)
display(teacher_summary)


### 3.1 EEG teacher diagnostics and frozen neural-target cache

The teacher is now frozen. For downstream experiments we cache only its token geometry and importance; the EEG network itself is not needed by the visual models.

In [ ]:

# ============================================================
# CELL 9 — TEACHER TEMPORAL DIAGNOSTICS + CACHED TARGETS
# ============================================================
def temporal_cosine(z):
    a=F.normalize(z[:,:-1],dim=-1); b=F.normalize(z[:,1:],dim=-1)
    return (a*b).sum(-1)

def entropy(p):
    p=p.clamp_min(1e-8); return -(p*p.log()).sum(-1)

teacher_diag={
    "test_adjacent_cosine_mean":float(temporal_cosine(test_teacher_tokens).mean()),
    "test_importance_entropy_mean":float(entropy(test_teacher_imp).mean()),
    "uniform_entropy":float(math.log(SEQ_LEN)),
    "test_macro_f1":teacher_test_metrics["macro_f1"],
}
with open(REPORT_ROOT/"stage31_teacher_temporal_diagnostics.json","w") as f: json.dump(teacher_diag,f,indent=2)
print(json.dumps(teacher_diag,indent=2))

def cache_teacher_targets(table,loader,split):
    tp=CACHE_ROOT/f"{split}_teacher_tokens_fp16.pt"
    ip=CACHE_ROOT/f"{split}_teacher_importance_fp16.pt"
    rp=CACHE_ROOT/f"{split}_teacher_relation_fp16.pt"
    if RESUME_PHASE3 and tp.exists() and ip.exists() and rp.exists():
        return torch.load(tp,map_location="cpu",weights_only=False),torch.load(ip,map_location="cpu",weights_only=False),torch.load(rp,map_location="cpu",weights_only=False)
    tok=[None]*len(table); imp=[None]*len(table); rel=[None]*len(table)
    with torch.no_grad():
        for xb,_,idx in loader:
            xb=xb.to(DEVICE)
            with autocast_context(): o=teacher(xb)
            z=o["tokens"].float().cpu(); a=o["importance"].float().cpu()
            r=F.normalize(z,dim=-1); r=torch.einsum("btd,bsd->bts",r,r)
            for j,k in enumerate(idx.numpy()):
                tok[int(k)]=z[j].half(); imp[int(k)]=a[j].half(); rel[int(k)]=r[j].half()
    tok,imp,rel=torch.stack(tok),torch.stack(imp),torch.stack(rel)
    torch.save(tok,tp); torch.save(imp,ip); torch.save(rel,rp)
    return tok,imp,rel

target_cache={}
for nm,tb,ld in [("train",train_seq,train_eeg_loader),("val",val_seq,val_eeg_loader),("test",test_seq,test_eeg_loader)]:
    tok,imp,rel=cache_teacher_targets(tb,ld,nm)
    target_cache[nm]={"tokens":tok,"importance":imp,"relation":rel}
    print(nm,tuple(tok.shape),tuple(imp.shape),tuple(rel.shape))

del teacher
if DEVICE.type=="cuda": torch.cuda.empty_cache()
print("PASS EEG teacher frozen; downstream stages use cached neural targets only.")


# Stage 3.2 — Visual temporal representation

Reconstruct the exact Phase-2 canonical Temporal Transformer from the frozen checkpoint. The intermediate projected token is `zV(t)`.

In [ ]:

# ============================================================
# CELL 10 — PHASE-2 VISUAL CORE
# ============================================================
class Phase2VisualCore(nn.Module):
    def __init__(self):
        super().__init__()
        self.input=nn.Linear(VISUAL_DIM,D_MODEL)
        self.pos=SinusoidalPositionalEncoding(D_MODEL,SEQ_LEN)
        layer=nn.TransformerEncoderLayer(d_model=D_MODEL,nhead=HEADS,dim_feedforward=FF_DIM,dropout=DROPOUT,activation="gelu",batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=LAYERS)
        self.norm=nn.LayerNorm(D_MODEL)
        self.classifier=nn.Linear(D_MODEL,NUM_CLASSES)
    def project_tokens(self,x): return self.input(x)
    def encode_tokens(self,z,positions=None,key_padding_mask=None):
        return self.norm(self.encoder(self.pos(z,positions),src_key_padding_mask=key_padding_mask))
    def forward(self,x):
        z=self.project_tokens(x); e=self.encode_tokens(z); return self.classifier(e.mean(1))

visual_core=Phase2VisualCore()
load_state(PHASE2_CHECKPOINT,visual_core,True)
visual_core=visual_core.to(DEVICE).eval()
q=torch.randn(2,SEQ_LEN,VISUAL_DIM,device=DEVICE)
assert visual_core.project_tokens(q).shape==(2,SEQ_LEN,D_MODEL)
assert visual_core(q).shape==(2,NUM_CLASSES)
print("PASS Phase-2 visual core loaded read-only.")


In [ ]:

# ============================================================
# CELL 11 — VISUAL DATASET + PHASE-2 BASELINE RUNTIME CHECK
# ============================================================
class VisualSequenceDataset(Dataset):
    def __init__(self,table):
        self.df=table.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        x=visual_cache[r.subject_id][int(r.visual_start):int(r.visual_start)+SEQ_LEN].float()
        x=(x-visual_mean)/visual_std
        return x,int(r.label_id),i

train_visual_ds=VisualSequenceDataset(train_seq); val_visual_ds=VisualSequenceDataset(val_seq); test_visual_ds=VisualSequenceDataset(test_seq)
vision_kwargs=dict(batch_size=VISION_BATCH,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=="cuda"))
train_visual_loader=DataLoader(train_visual_ds,shuffle=True,**vision_kwargs)
val_visual_loader=DataLoader(val_visual_ds,shuffle=False,**vision_kwargs)
test_visual_loader=DataLoader(test_visual_ds,shuffle=False,**vision_kwargs)

def eval_visual(model,loader,subject=False):
    model.eval(); ys=[]; ps=[]; ids=[]
    with torch.no_grad():
        for xb,yb,idx in loader:
            xb=xb.to(DEVICE)
            with autocast_context(): o=model(xb)
            ys.extend(yb.numpy().tolist()); ps.extend(o.argmax(1).cpu().numpy().tolist()); ids.extend(idx.numpy().tolist())
    m=compute_metrics(np.asarray(ys),np.asarray(ps))
    sm=None
    if subject:
        rr=[]
        for s,g in loader.dataset.df.groupby("subject_id"):
            pos=[i for i,k in enumerate(ids) if loader.dataset.df.iloc[int(k)].subject_id==s]
            yy=np.asarray([ys[j] for j in pos]); pp=np.asarray([ps[j] for j in pos])
            rr.append({"subject_id":s,**compute_metrics(yy,pp)})
        sm=pd.DataFrame(rr)
    return m,sm

phase2_clean_runtime,phase2_subject_metrics=eval_visual(visual_core,test_visual_loader,True)
phase2_subject_metrics.to_csv(REPORT_ROOT/"phase2_baseline_subject_metrics.csv",index=False)
pd.DataFrame([phase2_clean_runtime]).to_csv(REPORT_ROOT/"phase2_checkpoint_runtime.csv",index=False)
print("Phase-2 checkpoint runtime:",phase2_clean_runtime)


## 4. Transfer losses and cached-teacher visual dataset

Relation supervision matches EEG/visual temporal geometry through pairwise cosine similarity matrices. Importance supervision matches the teacher's temporal probability distribution.

In [ ]:

# ============================================================
# CELL 12 — RELATION / IMPORTANCE LOSSES + MULTIMODAL DATASET
# ============================================================
OFFDIAG=(~torch.eye(SEQ_LEN,dtype=torch.bool)).view(1,SEQ_LEN,SEQ_LEN)

def relation_matrix(z):
    z=F.normalize(z.float(),dim=-1)
    return torch.einsum("btd,bsd->bts",z,z)

def relation_loss(zv,target_relation):
    d=(relation_matrix(zv)-target_relation.float())**2
    return d.masked_select(OFFDIAG.to(d.device)).mean()

def importance_kl(student_logits,target_importance):
    p=target_importance.float().clamp_min(1e-8)
    p=p/p.sum(dim=1,keepdim=True)
    return F.kl_div(F.log_softmax(student_logits,dim=1),p,reduction="batchmean")

class CachedTeacherVisualDataset(Dataset):
    def __init__(self,table,targets):
        self.df=table.reset_index(drop=True); self.targets=targets
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        x=visual_cache[r.subject_id][int(r.visual_start):int(r.visual_start)+SEQ_LEN].float()
        x=(x-visual_mean)/visual_std
        return x,int(r.label_id),i,self.targets["importance"][i],self.targets["relation"][i]

train_multi_ds=CachedTeacherVisualDataset(train_seq,target_cache["train"])
val_multi_ds=CachedTeacherVisualDataset(val_seq,target_cache["val"])
test_multi_ds=CachedTeacherVisualDataset(test_seq,target_cache["test"])
train_multi_loader=DataLoader(train_multi_ds,shuffle=True,**vision_kwargs)
val_multi_loader=DataLoader(val_multi_ds,shuffle=False,**vision_kwargs)
test_multi_loader=DataLoader(test_multi_ds,shuffle=False,**vision_kwargs)
print("Rows:",len(train_multi_ds),len(val_multi_ds),len(test_multi_ds))


# Stage 3.3 — Neural temporal-relation transfer

Only classification + relation loss are active. No routing and no budget regularizer.

In [ ]:

# ============================================================
# CELL 13 — VISUAL NEURAL-TRANSFER MODEL + TRAINING
# ============================================================
class VisualNeuralModel(nn.Module):
    def __init__(self,init_checkpoint=None):
        super().__init__()
        self.core=Phase2VisualCore()
        if init_checkpoint is not None: load_state(init_checkpoint,self.core,True)
        self.importance_head=nn.Linear(D_MODEL,1)
    def forward_dense(self,x):
        z=self.core.project_tokens(x); e=self.core.encode_tokens(z)
        return {"pre_tokens":z,"tokens":e,"logits":self.core.classifier(e.mean(1))}
    def forward_with_importance(self,x):
        o=self.forward_dense(x); s=self.importance_head(o["pre_tokens"]).squeeze(-1)
        o["importance_logits"]=s; o["importance"]=torch.softmax(s,1); return o

def clone_state(model): return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def eval_transfer(model,loader):
    model.eval(); ys=[]; ps=[]; rels=[]; imps=[]
    with torch.no_grad():
        for xb,yb,_,timp,trel in loader:
            xb=xb.to(DEVICE); timp=timp.to(DEVICE); trel=trel.to(DEVICE)
            with autocast_context(): o=model.forward_with_importance(xb)
            ys.extend(yb.numpy().tolist()); ps.extend(o["logits"].argmax(1).cpu().numpy().tolist())
            rels.append(float(relation_loss(o["pre_tokens"],trel).cpu()))
            imps.append(float(importance_kl(o["importance_logits"],timp).cpu()))
    return compute_metrics(np.asarray(ys),np.asarray(ps)),float(np.mean(rels)),float(np.mean(imps))

def train_transfer(model,train_loader,val_loader,ckpt,epochs,patience_limit,use_relation,use_importance):
    model=model.to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=VISION_LR,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    best=-np.inf; best_state=None; patience=0; start=0; hist=[]
    if RESUME_PHASE3 and ckpt.exists():
        pl=torch.load(ckpt,map_location="cpu",weights_only=False); model.load_state_dict(pl["model_state"])
        if "optimizer_state" in pl: opt.load_state_dict(pl["optimizer_state"])
        if "scheduler_state" in pl: sched.load_state_dict(pl["scheduler_state"])
        best=float(pl.get("best_metric",-np.inf)); start=int(pl.get("epoch",0))+1; hist=pl.get("history",[])
        print("RESUME",ckpt.name,start)
    for ep in range(start,epochs):
        model.train(); total=0.; seen=0
        for xb,yb,_,timp,trel in train_loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); timp=timp.to(DEVICE); trel=trel.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast_context():
                o=model.forward_with_importance(xb)
                c=nn.functional.cross_entropy(o["logits"],yb,weight=class_weights.to(DEVICE))
                r=relation_loss(o["pre_tokens"],trel) if use_relation else xb.new_zeros(())
                i=importance_kl(o["importance_logits"],timp) if use_importance else xb.new_zeros(())
                loss=c+LAMBDA_RELATION*r+LAMBDA_IMPORTANCE*i
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step()
            total+=float(loss.detach().cpu())*len(yb); seen+=len(yb)
        sched.step()
        vm,vr,vi=eval_transfer(model,val_loader)
        row={"epoch":ep,"train_loss":total/max(seen,1),"val_macro_f1":vm["macro_f1"],"val_balanced_accuracy":vm["balanced_accuracy"],"val_relation_loss":vr,"val_importance_kl":vi}
        hist.append(row); pd.DataFrame(hist).to_csv(ckpt.with_suffix(".csv"),index=False)
        print(f"stage ep {ep+1:02d} | val F1 {vm['macro_f1']:.4f} | rel {vr:.4f} | imp {vi:.4f}")
        if vm["macro_f1"]>best+1e-6: best=vm["macro_f1"]; best_state=clone_state(model); patience=0
        else: patience+=1
        save_ckpt(ckpt,model,opt,ep,best,hist)
        if patience>=patience_limit: break
    if best_state is not None: model.load_state_dict(best_state)
    return model,hist

stage33_ckpt=MODEL_ROOT/"stage33_neural_relation.pt"
rel_model=VisualNeuralModel(PHASE2_CHECKPOINT)
rel_model,_=train_transfer(rel_model,train_multi_loader,val_multi_loader,stage33_ckpt,RELATION_EPOCHS,RELATION_PATIENCE,True,False)
rel_test,rel_test_loss,_=eval_transfer(rel_model,test_multi_loader)
with open(REPORT_ROOT/"stage33_report.json","w") as f: json.dump({"metrics":rel_test,"relation_loss":rel_test_loss},f,indent=2)
print("Stage 3.3:",rel_test, "relation loss",rel_test_loss)


# Stage 3.4 — Neural temporal-importance supervision

The EEG teacher's `aE(t)` now supervises a visual importance head. Routing is still absent.

In [ ]:

# ============================================================
# CELL 14 — STAGE 3.4 IMPORTANCE
# ============================================================
stage34_ckpt=MODEL_ROOT/"stage34_neural_importance.pt"
imp_model=VisualNeuralModel(PHASE2_CHECKPOINT)
imp_model.load_state_dict(rel_model.state_dict())
imp_model,_=train_transfer(imp_model,train_multi_loader,val_multi_loader,stage34_ckpt,IMPORTANCE_EPOCHS,IMPORTANCE_PATIENCE,True,True)
imp_test,imp_rel,imp_kl=eval_transfer(imp_model,test_multi_loader)
with open(REPORT_ROOT/"stage34_report.json","w") as f: json.dump({"metrics":imp_test,"relation_loss":imp_rel,"importance_kl":imp_kl},f,indent=2)
print("Stage 3.4:",imp_test,"relation",imp_rel,"importance KL",imp_kl)


# Stage 3.5 — Adaptive temporal routing

The router scores visual temporal tokens, applies exact top-k selection, preserves original temporal positions, and passes only selected tokens to the temporal Transformer at inference.

In [ ]:
# ============================================================
# CELL 15 — NEUROROUTE MODEL + ROUTER TRAINING
# ============================================================
class NeuroRouteModel(nn.Module):
    def __init__(self, source_model):
        super().__init__()
        self.core = Phase2VisualCore()
        self.core.load_state_dict(source_model.core.state_dict())
        self.route_head = nn.Linear(D_MODEL, 1)
        
        # Dynamically load from either importance_head or an existing route_head
        if hasattr(source_model, "route_head"):
            self.route_head.load_state_dict(source_model.route_head.state_dict())
        elif hasattr(source_model, "importance_head"):
            self.route_head.load_state_dict(source_model.importance_head.state_dict())
        else:
            raise AttributeError("source_model has neither route_head nor importance_head")

    def route_scores(self, x):
        z = self.core.project_tokens(x)
        return z, self.route_head(z).squeeze(-1)

    def _k(self, budget): 
        return max(1, int(round(SEQ_LEN * float(budget))))

    def training_forward(self, x, budget):
        z, s = self.route_scores(x)
        k = self._k(budget)
        soft = torch.sigmoid(s / ROUTER_SOFT_TEMPERATURE)
        idx = torch.topk(s, k=k, dim=1).indices
        hard = torch.zeros_like(s).scatter(1, idx, 1.0)
        ste = hard + soft - soft.detach()
        mask = hard < 0.5
        pos = torch.arange(SEQ_LEN, device=x.device).view(1, -1).expand(x.size(0), -1)
        e = self.core.encode_tokens(z * ste.unsqueeze(-1), positions=pos, key_padding_mask=mask)
        pooled = (e * hard.unsqueeze(-1)).sum(1) / hard.sum(1, keepdim=True).clamp_min(1.0)
        return {"z": z, "scores": s, "importance": torch.softmax(s, 1), "soft_keep": soft, "hard_keep": hard, "selected_idx": idx, "logits": self.core.classifier(pooled)}

    @torch.no_grad()
    def inference(self, x, budget):
        self.eval()
        z, s = self.route_scores(x)
        k = self._k(budget)
        idx = torch.sort(torch.topk(s, k=k, dim=1).indices, dim=1).values
        zsel = torch.gather(z, 1, idx.unsqueeze(-1).expand(-1, -1, z.shape[-1]))
        e = self.core.encode_tokens(zsel, positions=idx)
        return {"logits": self.core.classifier(e.mean(1)), "selected_idx": idx, "selected_k": k, "scores": s}

def eval_router(model, loader, budget):
    model.eval()
    ys = []
    ps = []
    with torch.no_grad():
        for xb, yb, *_ in loader:
            xb = xb.to(DEVICE)
            with autocast_context(): 
                o = model.inference(xb, budget)
            ys.extend(yb.numpy().tolist())
            ps.extend(o["logits"].argmax(1).cpu().numpy().tolist())
    return compute_metrics(np.asarray(ys), np.asarray(ps))

def train_router_fixed(model, train_loader, val_loader, ckpt):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=ROUTER_LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ROUTER_EPOCHS)
    best = -np.inf
    best_state = None
    patience = 0
    start = 0
    hist = []
    if RESUME_PHASE3 and ckpt.exists():
        pl = torch.load(ckpt, map_location="cpu", weights_only=False)
        model.load_state_dict(pl["model_state"])
        if "optimizer_state" in pl: 
            opt.load_state_dict(pl["optimizer_state"])
        best = float(pl.get("best_metric", -np.inf))
        start = int(pl.get("epoch", 0)) + 1
        hist = pl.get("history", [])
        print("RESUME", ckpt.name, start)
    for ep in range(start, ROUTER_EPOCHS):
        model.train()
        for xb, yb, idx, timp, trel in train_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE); timp = timp.to(DEVICE); trel = trel.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast_context():
                o = model.training_forward(xb, PRIMARY_BUDGET)
                c = F.cross_entropy(o["logits"], yb, weight=class_weights.to(DEVICE))
                r = relation_loss(o["z"], trel)
                i = importance_kl(o["scores"], timp)
                loss = c + LAMBDA_RELATION * r + LAMBDA_IMPORTANCE * i
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        sched.step()
        vm = eval_router(model, val_loader, PRIMARY_BUDGET)
        print(f"router ep {ep+1:02d} | val F1 {vm['macro_f1']:.4f}")
        hist.append({"epoch": ep, **{f"val_{k}": v for k, v in vm.items()}})
        if vm["macro_f1"] > best + 1e-6: 
            best = vm["macro_f1"]
            best_state = clone_state(model)
            patience = 0
        else: 
            patience += 1
        save_ckpt(ckpt, model, opt, ep, best, hist, {"primary_budget": PRIMARY_BUDGET})
        if patience >= ROUTER_PATIENCE: 
            break
    if best_state is not None: 
        model.load_state_dict(best_state)
    return model

stage35_ckpt = MODEL_ROOT / "stage35_router_50pct.pt"
router_model = NeuroRouteModel(imp_model)
router_model = train_router_fixed(router_model, train_multi_loader, val_multi_loader, stage35_ckpt)
print("Stage 3.5 primary-budget test:", eval_router(router_model, test_multi_loader, PRIMARY_BUDGET))

## 5. Temporal corruption operators

These reproduce the exact Phase-2 corruption definitions so robustness comparisons stay on the frozen benchmark.

In [ ]:

# ============================================================
# CELL 16 — EXACT PHASE-2 CORRUPTION FUNCTIONS
# ============================================================
def _random_keep_mask(B,T,rate,device): return torch.rand(B,T,device=device)>=rate
def corrupt_random_frame_drop(x,rate):
    B,T,D=x.shape; keep=_random_keep_mask(B,T,rate,x.device); out=torch.zeros_like(x)
    for b in range(B):
        z=x[b,keep[b]]; out[b,:min(T,len(z))]=z[:T]
    return out
def corrupt_burst_deletion(x,rate):
    B,T,D=x.shape; burst=max(1,int(math.ceil(rate*T))); out=torch.zeros_like(x)
    for b in range(B):
        if burst>=T: continue
        st=int(torch.randint(0,T-burst+1,(1,),device=x.device).item())
        z=torch.cat([x[b,:st],x[b,st+burst:]],0); out[b,:min(T,len(z))]=z[:T]
    return out
def corrupt_blackout(x,rate):
    B,T,D=x.shape; block=max(1,int(math.ceil(rate*T))); out=x.clone()
    for b in range(B):
        if block>=T: out[b]=0; continue
        st=int(torch.randint(0,T-block+1,(1,),device=x.device).item()); out[b,st:st+block]=0
    return out
def corrupt_temporal_jitter(x,rate):
    B,T,D=x.shape; rad=max(1,int(round(rate*T/2)))
    base=torch.arange(T,device=x.device).view(1,T).expand(B,T)
    off=torch.randint(-rad,rad+1,(B,T),device=x.device); idx=(base+off).clamp(0,T-1)
    return torch.gather(x,1,idx.unsqueeze(-1).expand(-1,-1,D))
def _resample(x,pos):
    B,T,D=x.shape; pos=pos.clamp(0,T-1); lo=pos.floor().long(); hi=pos.ceil().long(); a=(pos-lo.float()).unsqueeze(-1)
    lx=torch.gather(x,1,lo.unsqueeze(-1).expand(-1,-1,D)); hx=torch.gather(x,1,hi.unsqueeze(-1).expand(-1,-1,D))
    return lx*(1-a)+hx*a
def corrupt_irregular_sampling(x,rate):
    B,T,D=x.shape; base=torch.linspace(0,T-1,T,device=x.device).view(1,T).expand(B,T)
    pos=torch.sort((base+torch.randn(B,T,device=x.device)*(rate*0.75)).clamp(0,T-1),1).values
    return _resample(x,pos)
def corrupt_playback_speed(x,rate):
    B,T,D=x.shape; speed=1+(2*torch.rand(B,1,device=x.device)-1)*rate; c=(T-1)/2
    base=torch.arange(T,device=x.device).view(1,T).expand(B,T)
    return _resample(x,c+(base-c)*speed)
CORRUPTION_FUNCS={"random_frame_drop":corrupt_random_frame_drop,"burst_deletion":corrupt_burst_deletion,"temporal_jitter":corrupt_temporal_jitter,"irregular_sampling":corrupt_irregular_sampling,"playback_speed_change":corrupt_playback_speed,"contiguous_blackout":corrupt_blackout}
z=torch.randn(2,SEQ_LEN,VISUAL_DIM,device=DEVICE)
for fn in CORRUPTION_FUNCS.values(): assert fn(z,0.30).shape==z.shape
print("PASS six corruption families.")


# Stage 3.6 — Budget regularization

A single router is trained over all five budgets. The requested budget is part of the batch condition, while the top-k route remains exact at inference.

In [ ]:

# ============================================================
# CELL 17 — BUDGET-REGULARIZED ROUTER
# ============================================================
stage36_ckpt=MODEL_ROOT/"stage36_budget_router.pt"

def train_budget(model,ckpt):
    model=model.to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=ROUTER_LR,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=BUDGET_EPOCHS)
    best=-np.inf; best_state=None; patience=0; start=0; hist=[]
    if RESUME_PHASE3 and ckpt.exists():
        pl=torch.load(ckpt,map_location="cpu",weights_only=False); model.load_state_dict(pl["model_state"])
        if "optimizer_state" in pl: opt.load_state_dict(pl["optimizer_state"])
        best=float(pl.get("best_metric",-np.inf)); start=int(pl.get("epoch",0))+1; hist=pl.get("history",[])
        print("RESUME",ckpt.name,start)
    for ep in range(start,BUDGET_EPOCHS):
        model.train()
        for xb,yb,idx,timp,trel in train_multi_loader:
            budget=BUDGETS[(ep+int(idx[0]))%len(BUDGETS)]
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); timp=timp.to(DEVICE); trel=trel.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast_context():
                o=model.training_forward(xb,budget)
                c=F.cross_entropy(o["logits"],yb,weight=class_weights.to(DEVICE))
                r=relation_loss(o["z"],trel); i=importance_kl(o["scores"],timp)
                target=xb.new_full((len(yb),),float(budget))
                b=F.mse_loss(o["soft_keep"].mean(1),target)
                loss=c+LAMBDA_RELATION*r+LAMBDA_IMPORTANCE*i+LAMBDA_BUDGET*b
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step()
        sched.step()
        vm=eval_router(model,val_multi_loader,PRIMARY_BUDGET)
        hist.append({"epoch":ep,**{f"val_{k}":v for k,v in vm.items()}})
        print(f"budget ep {ep+1:02d} | val F1 {vm['macro_f1']:.4f}")
        if vm["macro_f1"]>best+1e-6: best=vm["macro_f1"]; best_state=clone_state(model); patience=0
        else: patience+=1
        save_ckpt(ckpt,model,opt,ep,best,hist,{"budgets":BUDGETS,"primary_budget":PRIMARY_BUDGET})
        if patience>=BUDGET_PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

budget_model=NeuroRouteModel(router_model)
budget_model=train_budget(budget_model,stage36_ckpt) if RUN_STAGE_36_BUDGET else budget_model
if not RUN_STAGE_36_BUDGET: load_state(stage36_ckpt,budget_model,True)

sweep=[]
for b in BUDGETS:
    m=eval_router(budget_model,test_multi_loader,b)
    sweep.append({"budget":b,"tokens":max(1,int(round(SEQ_LEN*b))),**m})
budget_df=pd.DataFrame(sweep)
budget_df.to_csv(REPORT_ROOT/"stage36_budget_sweep_test.csv",index=False)
display(budget_df)


# Stage 3.6b — Robustness-aware NeuroRoute

A final incremental fine-tuning stage adds prediction-consistency under temporal corruption while retaining relation, importance, and budget terms.

In [ ]:

# ============================================================
# CELL 18 — ROBUSTNESS-AWARE ROUTER
# ============================================================
stage36b_ckpt=MODEL_ROOT/"stage36b_robust_router.pt"

def symmetric_kl(p_logits,q_logits):
    p_log=F.log_softmax(p_logits,1); q_log=F.log_softmax(q_logits,1)
    return 0.5*(F.kl_div(p_log,q_log.exp(),reduction="batchmean")+F.kl_div(q_log,p_log.exp(),reduction="batchmean"))

def train_robust(model,ckpt):
    model=model.to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=ROUTER_LR,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=ROBUST_EPOCHS)
    best=-np.inf; best_state=None; patience=0; start=0; hist=[]
    if RESUME_PHASE3 and ckpt.exists():
        pl=torch.load(ckpt,map_location="cpu",weights_only=False); model.load_state_dict(pl["model_state"])
        if "optimizer_state" in pl: opt.load_state_dict(pl["optimizer_state"])
        best=float(pl.get("best_metric",-np.inf)); start=int(pl.get("epoch",0))+1; hist=pl.get("history",[])
        print("RESUME",ckpt.name,start)
    for ep in range(start,ROBUST_EPOCHS):
        model.train()
        for xb,yb,idx,timp,trel in train_multi_loader:
            budget=BUDGETS[(ep+int(idx[0]))%len(BUDGETS)]
            cname=CORRUPTION_NAMES[(ep+int(idx[0]))%len(CORRUPTION_NAMES)]
            sev=[0.15,0.30,0.50][(ep+int(idx[0]))%3]
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); timp=timp.to(DEVICE); trel=trel.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast_context():
                clean=model.training_forward(xb,budget)
                corrupt_x=CORRUPTION_FUNCS[cname](xb,sev)
                corr=model.training_forward(corrupt_x,budget)
                c=F.cross_entropy(clean["logits"],yb,weight=class_weights.to(DEVICE))
                r=relation_loss(clean["z"],trel); i=importance_kl(clean["scores"],timp)
                b=F.mse_loss(clean["soft_keep"].mean(1),xb.new_full((len(yb),),float(budget)))
                rb=symmetric_kl(clean["logits"].detach(),corr["logits"])
                loss=c+LAMBDA_RELATION*r+LAMBDA_IMPORTANCE*i+LAMBDA_BUDGET*b+LAMBDA_ROBUST*rb
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step()
        sched.step()
        vm=eval_router(model,val_multi_loader,PRIMARY_BUDGET)
        hist.append({"epoch":ep,**{f"val_{k}":v for k,v in vm.items()}})
        print(f"robust ep {ep+1:02d} | val F1 {vm['macro_f1']:.4f}")
        if vm["macro_f1"]>best+1e-6: best=vm["macro_f1"]; best_state=clone_state(model); patience=0
        else: patience+=1
        save_ckpt(ckpt,model,opt,ep,best,hist,{"primary_budget":PRIMARY_BUDGET,"lambda_robust":LAMBDA_ROBUST})
        if patience>=ROBUST_PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

robust_model=NeuroRouteModel(budget_model)
robust_model=train_robust(robust_model,stage36b_ckpt) if RUN_STAGE_36_ROBUSTNESS else robust_model
if not RUN_STAGE_36_ROBUSTNESS: load_state(stage36b_ckpt,robust_model,True)


## 6. Temporal computation accounting

The visual embeddings are already precomputed. Efficiency therefore refers to the **temporal Transformer computation after routing**.

In [ ]:

# ============================================================
# CELL 19 — TEMPORAL FLOPs + GPU LATENCY
# ============================================================
def estimate_temporal_flops(T):
    d=D_MODEL; ff=FF_DIM; L=LAYERS; c=NUM_CLASSES
    input_proj=2*T*VISUAL_DIM*d
    qkv=2*T*d*(3*d); out=2*T*d*d
    attn=4*T*T*d
    ffn=2*T*d*ff+2*T*ff*d
    clf=2*d*c
    return int(input_proj+L*(qkv+out+attn+ffn)+clf)

def latency_ms(model,budget,repeats=100,warmup=20):
    x=torch.randn(1,SEQ_LEN,VISUAL_DIM,device=DEVICE); model.eval()
    with torch.no_grad():
        for _ in range(warmup): model.inference(x,budget)
    if DEVICE.type=="cuda":
        torch.cuda.synchronize(); a=torch.cuda.Event(enable_timing=True); b=torch.cuda.Event(enable_timing=True)
        a.record()
        with torch.no_grad():
            for _ in range(repeats): model.inference(x,budget)
        b.record(); torch.cuda.synchronize(); return float(a.elapsed_time(b)/repeats)
    t=time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats): model.inference(x,budget)
    return float((time.perf_counter()-t)*1000/repeats)

dense_flops=estimate_temporal_flops(SEQ_LEN)
compute_rows=[]
for b in BUDGETS:
    T=max(1,int(round(SEQ_LEN*b)))
    row={"budget":b,"tokens":T,"estimated_temporal_flops":estimate_temporal_flops(T)}
    row["flop_reduction"]=1-row["estimated_temporal_flops"]/dense_flops
    if DEVICE.type=="cuda": row["latency_ms"]=latency_ms(robust_model,b)
    compute_rows.append(row)
compute_df=pd.DataFrame(compute_rows)
compute_df.to_csv(REPORT_ROOT/"temporal_compute_budget_sweep.csv",index=False)
display(compute_df)


# 7. Primary 50% robustness benchmark

The Phase-2 corruption benchmark is loaded from the frozen Phase-2 output, while NeuroRoute is evaluated with the exact same corruption families, severities, and deterministic seeds.

In [ ]:

# ============================================================
# CELL 20 — PRIMARY CORRUPTION BENCHMARK
# ============================================================
if PHASE2_CORRUPTIONS is None: raise FileNotFoundError("Frozen Phase-2 corruption_benchmark.csv missing.")
p2c=pd.read_csv(PHASE2_CORRUPTIONS)
p2c=p2c[p2c.model.astype(str)=="temporal_transformer"].copy()
expected_rows=len(CORRUPTION_NAMES)*len(CORRUPTION_SEVERITIES)
if len(p2c)!=expected_rows: raise RuntimeError(f"Frozen Phase-2 corruption rows: expected {expected_rows}, got {len(p2c)}")

def eval_route_corruptions(model,loader,budget):
    rows=[]
    for cname in CORRUPTION_NAMES:
        for sev in CORRUPTION_SEVERITIES:
            seed=SEED+int(sev*1000)+CORRUPTION_NAMES.index(cname)*10000
            torch.manual_seed(seed)
            if DEVICE.type=="cuda": torch.cuda.manual_seed_all(seed)
            ys=[]; ps=[]
            for xb,yb,*_ in loader:
                xb=CORRUPTION_FUNCS[cname](xb.to(DEVICE),sev)
                with torch.no_grad():
                    with autocast_context(): o=model.inference(xb,budget)
                ys.extend(yb.numpy().tolist()); ps.extend(o["logits"].argmax(1).cpu().numpy().tolist())
            rows.append({"model":"neuroroute_50pct","budget":budget,"corruption":cname,"severity":sev,**compute_metrics(np.asarray(ys),np.asarray(ps))})
            print(cname,sev,rows[-1]["macro_f1"])
    return pd.DataFrame(rows)

final_corr=eval_route_corruptions(robust_model,test_multi_loader,PRIMARY_BUDGET)
p2c2=p2c[["corruption","severity","macro_f1","balanced_accuracy"]].copy()
p2c2["model"]="phase2_temporal_transformer"; p2c2["budget"]=1.0
corr_compare=pd.concat([p2c2,final_corr],ignore_index=True)
final_corr.to_csv(REPORT_ROOT/"stage36b_primary_corruption.csv",index=False)
corr_compare.to_csv(REPORT_ROOT/"stage36b_robustness_comparison.csv",index=False)
display(corr_compare.head(36))


# 8. EEG removal test

The deployable model is a visual-only copy of the final routed network. Its state dictionary is audited for EEG-specific components.

In [ ]:

# ============================================================
# CELL 21 — VISUAL-ONLY DEPLOYMENT
# ============================================================
class DeployableNeuroRoute(nn.Module):
    def __init__(self,trained):
        super().__init__()
        self.core=Phase2VisualCore(); self.core.load_state_dict(trained.core.state_dict())
        self.route_head=nn.Linear(D_MODEL,1); self.route_head.load_state_dict(trained.route_head.state_dict())
    @torch.no_grad()
    def forward(self,x,budget=PRIMARY_BUDGET):
        self.eval()
        z=self.core.project_tokens(x); s=self.route_head(z).squeeze(-1)
        k=max(1,int(round(SEQ_LEN*budget)))
        idx=torch.sort(torch.topk(s,k=k,dim=1).indices,1).values
        zsel=torch.gather(z,1,idx.unsqueeze(-1).expand(-1,-1,z.shape[-1]))
        e=self.core.encode_tokens(zsel,positions=idx)
        return self.core.classifier(e.mean(1)),idx

deploy_model=DeployableNeuroRoute(robust_model).to(DEVICE).eval()
for n in deploy_model.state_dict():
    if any(x in n.lower() for x in ["eeg","teacher","neural_target"]):
        raise RuntimeError(f"EEG-specific parameter leaked: {n}")

ys=[]; ps=[]
for xb,yb,*_ in test_visual_loader:
    xb=xb.to(DEVICE)
    with autocast_context(): logits,_=deploy_model(xb,PRIMARY_BUDGET)
    ys.extend(yb.numpy().tolist()); ps.extend(logits.argmax(1).cpu().numpy().tolist())
deployment_metrics=compute_metrics(np.asarray(ys),np.asarray(ps))

deployment_ckpt=DEPLOY_ROOT/"neuroroute_visual_only.pt"
torch.save({
    "state_dict":deploy_model.state_dict(),"architecture":"DeployableNeuroRoute",
    "visual_dim":VISUAL_DIM,"sequence_length":SEQ_LEN,"primary_budget":PRIMARY_BUDGET,
    "eeg_required":False,"eeg_preprocessing_required":False,"eeg_encoder_required":False,
},deployment_ckpt)
deployment_report={"test_metrics":deployment_metrics,"checkpoint":str(deployment_ckpt),"eeg_required":False,"eeg_preprocessing_required":False,"eeg_encoder_required":False}
with open(REPORT_ROOT/"stage37_eeg_removal_report.json","w") as f: json.dump(deployment_report,f,indent=2)
print(json.dumps(deployment_report,indent=2))


# 9. Critical negative controls

Four training controls are used at the same 50% route:
1. random EEG teacher targets,
2. wrong-subject EEG targets,
3. temporally shifted EEG targets,
4. no meaningful neural supervision.

The frozen Phase-2 visual baseline remains Control 1.

In [ ]:

# ============================================================
# CELL 22 — NEGATIVE CONTROL TRAINING
# ============================================================
def build_wrong_subject_mapping(table,seed=SEED+777):
    rng=np.random.default_rng(seed); s=table.subject_id.to_numpy(); mp=np.empty(len(s),dtype=np.int64)
    for i in range(len(s)): mp[i]=int(rng.choice(np.where(s!=s[i])[0]))
    return mp

def train_control(init_model,mode,name):
    model=NeuroRouteModel(init_model).to(DEVICE)
    ckpt=CONTROL_ROOT/f"{name}.pt"
    opt=torch.optim.AdamW(model.parameters(),lr=ROUTER_LR,weight_decay=WEIGHT_DECAY)
    best=-np.inf; best_state=None; patience=0
    mp=None
    if mode=="random": mp=np.random.default_rng(SEED+901).permutation(len(train_seq))
    if mode=="wrong_subject": mp=build_wrong_subject_mapping(train_seq)
    for ep in range(CONTROL_EPOCHS):
        model.train()
        for xb,yb,idx,timp,trel in train_multi_loader:
            ids=idx.numpy()
            if mode=="random":
                ti=target_cache["train"]["importance"][mp[ids]]
                tr=target_cache["train"]["relation"][mp[ids]]
            elif mode=="wrong_subject":
                ti=target_cache["train"]["importance"][mp[ids]]
                tr=target_cache["train"]["relation"][mp[ids]]
            elif mode=="shifted":
                ti=torch.roll(target_cache["train"]["importance"][ids],2,1)
                tr=torch.roll(torch.roll(target_cache["train"]["relation"][ids],2,1),2,2)
            else:
                ti=timp; tr=trel

            xb=xb.to(DEVICE); yb=yb.to(DEVICE); ti=ti.to(DEVICE); tr=tr.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast_context():
                o=model.training_forward(xb,PRIMARY_BUDGET)
                cls=F.cross_entropy(o["logits"],yb,weight=class_weights.to(DEVICE))
                rel=relation_loss(o["z"],tr) if mode!="none" else xb.new_zeros(())
                imp=importance_kl(o["scores"],ti) if mode!="none" else xb.new_zeros(())
                bud=F.mse_loss(o["soft_keep"].mean(1),xb.new_full((len(yb),),PRIMARY_BUDGET))
                loss=cls+LAMBDA_RELATION*rel+LAMBDA_IMPORTANCE*imp+LAMBDA_BUDGET*bud
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step()

        vm=eval_router(model,val_multi_loader,PRIMARY_BUDGET)
        print(f"{name} ep {ep+1:02d} | val F1 {vm['macro_f1']:.4f}")
        if vm["macro_f1"]>best+1e-6: best=vm["macro_f1"]; best_state=clone_state(model); patience=0
        else: patience+=1
        if patience>=CONTROL_PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

control_specs=[("random_eeg","random"),("wrong_subject_eeg","wrong_subject"),("shifted_eeg","shifted"),("no_neural_supervision","none")]
control_rows=[]
control_models={}
if RUN_NEGATIVE_CONTROLS:
    for name,mode in control_specs:
        cm=train_control(imp_model,mode,name)
        control_models[name]=cm
        m=eval_router(cm,test_multi_loader,PRIMARY_BUDGET)
        control_rows.append({"control":name,"mode":mode,**m})
controls_df=pd.DataFrame(control_rows)
controls_df.to_csv(REPORT_ROOT/"negative_control_clean.csv",index=False)
display(controls_df)


In [ ]:

# ============================================================
# CELL 23 — NEGATIVE-CONTROL ROBUSTNESS @ 50% / 50% SEVERITY
# ============================================================
control_robust_rows=[]
if RUN_NEGATIVE_CONTROLS:
    for name,model in control_models.items():
        for cname in CORRUPTION_NAMES:
            seed=SEED+500+CORRUPTION_NAMES.index(cname)*10000
            torch.manual_seed(seed)
            if DEVICE.type=="cuda": torch.cuda.manual_seed_all(seed)
            ys=[]; ps=[]
            for xb,yb,*_ in test_multi_loader:
                xb=CORRUPTION_FUNCS[cname](xb.to(DEVICE),0.50)
                with torch.no_grad():
                    with autocast_context(): o=model.inference(xb,PRIMARY_BUDGET)
                ys.extend(yb.numpy().tolist()); ps.extend(o["logits"].argmax(1).cpu().numpy().tolist())
            control_robust_rows.append({"control":name,"corruption":cname,"severity":0.50,**compute_metrics(np.asarray(ys),np.asarray(ps))})
controls_robust_df=pd.DataFrame(control_robust_rows)
controls_robust_df.to_csv(REPORT_ROOT/"negative_control_robustness_50pct.csv",index=False)
display(controls_robust_df)


## 10. Subject-level paired analysis

The cross-subject test set contains eight held-out subjects. We compute subject-level macro-F1 and a paired bootstrap confidence interval for NeuroRoute minus the Phase-2 baseline.

In [ ]:

# ============================================================
# CELL 24 — SUBJECT-LEVEL CLEAN ANALYSIS + BOOTSTRAP
# ============================================================
def subject_metrics_visual(model,loader,budget=None):
    per={}
    model.eval()
    with torch.no_grad():
        for xb,yb,idx in loader:
            xb=xb.to(DEVICE)
            with autocast_context():
                if budget is None: logits=model(xb)
                else: logits=model(xb,budget)[0]
            pred=logits.argmax(1).cpu().numpy(); true=yb.numpy()
            for j,gidx in enumerate(idx.numpy()):
                s=loader.dataset.df.iloc[int(gidx)].subject_id
                per.setdefault(s,[[],[]]); per[s][0].append(int(true[j])); per[s][1].append(int(pred[j]))
    return pd.DataFrame([{"subject_id":s,**compute_metrics(np.asarray(v[0]),np.asarray(v[1]))} for s,v in per.items()])

nr_subject=subject_metrics_visual(deploy_model,test_visual_loader,PRIMARY_BUDGET)
nr_subject.to_csv(REPORT_ROOT/"neuroroute_subject_clean_metrics.csv",index=False)

paired=phase2_subject_metrics.merge(nr_subject,on="subject_id",suffixes=("_phase2","_neuroroute"))
delta=(paired["macro_f1_neuroroute"]-paired["macro_f1_phase2"]).to_numpy()
rng=np.random.default_rng(SEED+12345)
boots=np.asarray([rng.choice(delta,size=len(delta),replace=True).mean() for _ in range(BOOTSTRAP_ITERS)])
ci_lo,ci_hi=np.quantile(boots,[0.025,0.975])
bootstrap_report={
    "subjects":int(len(delta)),
    "mean_subject_macro_f1_delta":float(delta.mean()),
    "bootstrap_ci95_low":float(ci_lo),
    "bootstrap_ci95_high":float(ci_hi),
    "ci_excludes_zero":bool(ci_lo>0 or ci_hi<0),
}
with open(REPORT_ROOT/"subject_paired_bootstrap_report.json","w") as f: json.dump(bootstrap_report,f,indent=2)
display(paired)
print(bootstrap_report)


# 11. Final Phase-3 gate

The gate is deliberately transparent. A result can be **INCONCLUSIVE** when the system ran correctly but the scientific claim is not supported.

In [ ]:

# ============================================================
# CELL 25 — FINAL PHASE-3 GATE
# ============================================================
phase2_clean=phase2_clean_runtime
neuroroute_clean=deployment_metrics

phase2_robust_mean=float(p2c["macro_f1"].mean())
neuroroute_robust_mean=float(final_corr["macro_f1"].mean())
robustness_gain=neuroroute_robust_mean-phase2_robust_mean
primary_compute=compute_df[compute_df.budget==PRIMARY_BUDGET].iloc[0]

clean_ok=neuroroute_clean["macro_f1"] >= phase2_clean["macro_f1"]-CLEAN_TOL_ABS
robust_ok=robustness_gain >= ROBUSTNESS_GAIN_MIN
compute_ok=bool(primary_compute["estimated_temporal_flops"] < dense_flops)

if not controls_df.empty:
    control_gap=float(neuroroute_clean["macro_f1"]-controls_df["macro_f1"].max())
    control_ok=control_gap>=CONTROL_WEAKENING_MIN
else:
    control_gap=float("nan"); control_ok=False

deployment_ok=(deployment_report["eeg_required"] is False and deployment_report["eeg_preprocessing_required"] is False and deployment_report["eeg_encoder_required"] is False and deployment_ckpt.exists())
split_ok=(len(train_subjects)==24 and len(val_subjects)==8 and len(test_subjects)==8 and not(set(train_subjects)&set(val_subjects)|set(train_subjects)&set(test_subjects)|set(val_subjects)&set(test_subjects)))
teacher_ok=teacher_test_metrics["macro_f1"]>1/NUM_CLASSES

checks=OrderedDict([
    ("phase2_subject_split_locked",split_ok),
    ("eeg_teacher_above_uniform_macro_f1_chance",teacher_ok),
    ("primary_50pct_clean_recognition_preserved",clean_ok),
    ("primary_50pct_corruption_improved",robust_ok),
    ("primary_50pct_temporal_compute_reduced",compute_ok),
    ("neural_control_effect_observed",control_ok),
    ("eeg_removed_from_deployment",deployment_ok),
])

if not split_ok or not deployment_ok: status="FAIL"
elif all(checks.values()): status="PASS"
else: status="INCONCLUSIVE"

gate={
    "phase":"phase_3_neuroroute","status":status,"checks":checks,
    "primary_budget":PRIMARY_BUDGET,
    "phase2_clean":phase2_clean,"neuroroute_clean":neuroroute_clean,
    "phase2_mean_corruption_macro_f1":phase2_robust_mean,
    "neuroroute_mean_corruption_macro_f1":neuroroute_robust_mean,
    "robustness_gain":robustness_gain,
    "dense_temporal_flops":int(dense_flops),
    "primary_temporal_flops":int(primary_compute["estimated_temporal_flops"]),
    "primary_flop_reduction":float(primary_compute["flop_reduction"]),
    "control_gap":control_gap,
    "teacher_test_metrics":teacher_test_metrics,
    "deployment_report":deployment_report,
    "bootstrap":bootstrap_report,
}
with open(REPORT_ROOT/"phase3_gate_report.json","w") as f: json.dump(gate,f,indent=2,default=lambda x: bool(x) if isinstance(x,np.bool_) else x)

lines=["EgoBrain PHASE 3 — NEUROROUTE FINAL GATE","="*78,f"STATUS: {status}","",f"Primary budget: {PRIMARY_BUDGET:.1%} ({int(round(SEQ_LEN*PRIMARY_BUDGET))}/{SEQ_LEN} tokens)",""]
for k,v in checks.items(): lines.append(f"{'PASS' if v else 'FAIL':5s}  {k}")
lines += [
    "","Clean test:",
    f"  Phase-2 macro-F1:    {phase2_clean['macro_f1']:.6f}",
    f"  NeuroRoute macro-F1: {neuroroute_clean['macro_f1']:.6f}",
    "",
    "Robustness:",
    f"  Phase-2 mean corruption macro-F1:    {phase2_robust_mean:.6f}",
    f"  NeuroRoute mean corruption macro-F1: {neuroroute_robust_mean:.6f}",
    f"  Gain: {robustness_gain:+.6f}",
    "",
    "Temporal computation:",
    f"  Dense 100% FLOPs: {int(dense_flops):,}",
    f"  50% route FLOPs:  {int(primary_compute['estimated_temporal_flops']):,}",
    f"  Reduction:        {primary_compute['flop_reduction']:.2%}",
    "",
    "EEG removal: inference=no EEG; preprocessing=no EEG; encoder=no EEG.",
]
if status=="PASS": lines.append("PHASE 3 CLEARED — predefined NeuroRoute evidence established.")
elif status=="INCONCLUSIVE": lines.append("PHASE 3 INCONCLUSIVE — pipeline completed but one or more claims are not established.")
else: lines.append("PHASE 3 FAILED — inspect reproducibility/deployment failures.")
with open(REPORT_ROOT/"phase3_gate_report.txt","w") as f: f.write("\n".join(lines))
print("\n".join(lines))


## 12. Paper-oriented figures

In [ ]:

# ============================================================
# CELL 26 — FIGURES
# ============================================================
fig,ax=plt.subplots(figsize=(7,5))
ax.plot(budget_df["budget"]*100,budget_df["macro_f1"],marker="o",label="NeuroRoute")
ax.axhline(phase2_clean["macro_f1"],ls="--",label="Phase-2 dense baseline")
ax.set(xlabel="Retained temporal tokens (%)",ylabel="Test Macro-F1",title="Recognition vs temporal budget")
ax.grid(True,alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(FIG_ROOT/"01_budget_vs_macroF1.png",dpi=180); plt.close(fig)

fig,ax=plt.subplots(figsize=(7,5))
ax.plot(compute_df["tokens"],compute_df["estimated_temporal_flops"]/1e6,marker="o")
ax.set(xlabel="Retained temporal tokens",ylabel="Estimated temporal FLOPs (M)",title="Temporal computation vs budget")
ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(FIG_ROOT/"02_budget_vs_flops.png",dpi=180); plt.close(fig)

fig,ax=plt.subplots(figsize=(9,5))
for cname in CORRUPTION_NAMES:
    p=p2c2[p2c2.corruption==cname].sort_values("severity"); n=final_corr[final_corr.corruption==cname].sort_values("severity")
    ax.plot(p.severity*100,p.macro_f1,ls="--",alpha=.65,label=f"P2 {cname}")
    ax.plot(n.severity*100,n.macro_f1,marker="o",label=f"NR {cname}")
ax.set(xlabel="Severity (%)",ylabel="Macro-F1",title="Temporal robustness — Phase 2 vs NeuroRoute @ 50%")
ax.grid(True,alpha=.2); ax.legend(fontsize=7,ncol=2); fig.tight_layout(); fig.savefig(FIG_ROOT/"03_robustness_50pct.png",dpi=180); plt.close(fig)

if not controls_df.empty:
    plot=pd.concat([pd.DataFrame([{"control":"Phase-2 baseline","macro_f1":phase2_clean["macro_f1"]},{"control":"NeuroRoute aligned","macro_f1":neuroroute_clean["macro_f1"]}]),controls_df[["control","macro_f1"]]])
    fig,ax=plt.subplots(figsize=(8,5)); ax.bar(plot.control,plot.macro_f1); ax.set_ylabel("Test Macro-F1"); ax.set_title("Neural negative controls"); ax.tick_params(axis="x",rotation=25); fig.tight_layout(); fig.savefig(FIG_ROOT/"04_negative_controls.png",dpi=180); plt.close(fig)

fig,ax=plt.subplots(figsize=(7,4))
ax.plot(np.arange(1,SEQ_LEN+1),test_teacher_imp.float().mean(0).numpy(),marker="o")
ax.set(xlabel="Second in 8-s window",ylabel="Mean EEG teacher importance",title="EEG teacher temporal importance")
ax.set_xticks(np.arange(1,SEQ_LEN+1)); ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(FIG_ROOT/"05_eeg_teacher_importance.png",dpi=180); plt.close(fig)
print("Figures:",FIG_ROOT)


# 13. Package all Phase-3 artifacts

The package contains Phase-3 checkpoints, cached teacher targets, reports, figures, negative-control outputs, and the visual-only deployment checkpoint. Mounted Phase-1/Phase-2 raw inputs are not copied.

In [ ]:

# ============================================================
# CELL 27 — PACKAGE
# ============================================================
PACKAGE_BASE=OUT_ROOT.parent/"egobrain_phase3_neuroroute_artifacts"
ZIP_PATH=OUT_ROOT.parent/"egobrain_phase3_neuroroute_artifacts.zip"
if PACKAGE_BASE.exists(): shutil.rmtree(PACKAGE_BASE)
PACKAGE_BASE.mkdir(parents=True)

for src in [MODEL_ROOT,CACHE_ROOT,REPORT_ROOT,FIG_ROOT,CONTROL_ROOT,DEPLOY_ROOT]:
    if src.exists(): shutil.copytree(src,PACKAGE_BASE/src.name)

manifest={
    "phase":"phase_3_neuroroute","seed":SEED,"sequence_length":SEQ_LEN,"visual_dim":VISUAL_DIM,"eeg_dim":EEG_DIM,
    "num_classes":NUM_CLASSES,"train_subjects":train_subjects,"val_subjects":val_subjects,"test_subjects":test_subjects,
    "primary_budget":PRIMARY_BUDGET,"budgets":BUDGETS,"corruptions":CORRUPTION_NAMES,"severities":CORRUPTION_SEVERITIES,
    "phase2_checkpoint":str(PHASE2_CHECKPOINT),"phase2_split":str(PHASE2_SPLIT),"phase1_labeled_index":str(LABELED_INDEX),
    "visual_root":str(VISUAL_ROOT),"eeg_root":str(EEG_ROOT),"deployment_checkpoint":str(deployment_ckpt),
}
with open(PACKAGE_BASE/"phase3_manifest.json","w") as f: json.dump(manifest,f,indent=2)

hash_rows=[]
for p in sorted(PACKAGE_BASE.rglob("*")):
    if p.is_file():
        h=hashlib.sha256()
        with p.open("rb") as f:
            for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
        hash_rows.append({"relative_path":str(p.relative_to(PACKAGE_BASE)),"sha256":h.hexdigest(),"bytes":p.stat().st_size})
pd.DataFrame(hash_rows).to_csv(PACKAGE_BASE/"artifact_sha256.csv",index=False)

if ZIP_PATH.exists(): ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH.with_suffix("")),"zip",root_dir=PACKAGE_BASE)
print("ZIP:",ZIP_PATH)
print("Size MB:",round(ZIP_PATH.stat().st_size/1024**2,2))
print("Deployment checkpoint:",deployment_ckpt)


## Reproduction record

The completed route run writes model checkpoints, clean/robustness/negative-control reports, deployment audit, subject bootstrap analysis, figures, and `phase3_gate_report.{json,txt}`. The expected scientific conclusion from the recorded run is **INCONCLUSIVE**.